# Análisis de Ventas — Superstore Dataset
Dataset: [Kaggle - Superstore Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final/data)  
Herramientas: Python, Pandas, SQLite, SQL


## Introducción

Este proyecto consiste en un análisis exploratorio de datos (EDA) aplicado al dataset público *Sample Superstore* de Kaggle, que contiene registros de ventas de una tienda de retail en Estados Unidos entre los años 2014 y 2017, con un total de 9.994 órdenes de compra.

El objetivo es simular el trabajo de un analista de datos en un contexto de negocio real: extraer, estructurar y consultar los datos usando SQL, identificar patrones relevantes, y traducir esos hallazgos en conclusiones accionables para la toma de decisiones.

Las herramientas utilizadas fueron Python (pandas) para la carga y limpieza del dataset, SQLite como motor de base de datos relacional, y SQL para todas las consultas de análisis.



In [6]:
import sqlite3
import pandas as pd

# Conectar a la base de datos
conn = sqlite3.connect('Datos\superstore.db')
print('Conexión exitosa a superstore.db')

Conexión exitosa a superstore.db


## 1. Ventas y ganancia por categoría

In [7]:
query = '''
SELECT 
    category,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY category
ORDER BY total_ventas DESC;
'''

df_categoria = pd.read_sql(query, conn)
df_categoria

,category,total_ventas,total_ganancia,margen_pct
0,Technology,836154.03,145454.95,17.40
1,Furniture,741999.80,18451.27,2.49
2,Office Supplies,719047.03,122490.80,17.04


**Hallazgo:** Technology lidera en ventas y tiene el mejor margen (17.4%). Furniture vende un volumen similar pero su margen es de apenas 2.49%.

## 2. Ventas y ganancia por sub-categoría

In [8]:
query = '''
SELECT 
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY category, sub_category
ORDER BY margen_pct ASC;
'''

df_subcategoria = pd.read_sql(query, conn)
df_subcategoria

,category,sub_category,total_ventas,total_ganancia,margen_pct
0,Furniture,Tables,206965.53,-17725.48,-8.56
1,Furniture,Bookcases,114880.00,-3472.56,-3.02
2,Office Supplies,Supplies,46673.54,-1189.10,-2.55
3,Technology,Machines,189238.63,3384.76,1.79
4,Furniture,Chairs,328449.10,26590.17,8.10
5,Office Supplies,Storage,223843.61,21278.83,9.51
6,Technology,Phones,330007.05,44515.73,13.49
7,Furniture,Furnishings,91705.16,13059.14,14.24
8,Office Supplies,Binders,203412.73,30221.76,14.86
9,Office Supplies,Appliances,107532.16,18138.01,16.87


**Hallazgo:** Tres sub-categorías tienen ganancia negativa: Tables (-8.56%), Bookcases (-3.02%) y Supplies (-2.55%). Tables es el caso más crítico.

## 3. Descuento promedio vs ganancia por sub-categoría

In [9]:
query = '''
SELECT 
    sub_category,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia_total,
    COUNT(*) AS num_ordenes
FROM orders
WHERE sub_category IN ('Tables', 'Bookcases', 'Supplies', 'Copiers', 'Paper')
GROUP BY sub_category
ORDER BY descuento_promedio DESC;
'''

df_descuento_sub = pd.read_sql(query, conn)
df_descuento_sub

,sub_category,descuento_promedio,ganancia_total,num_ordenes
0,Tables,0.261,-17725.48,319
1,Bookcases,0.211,-3472.56,228
2,Copiers,0.162,55617.82,68
3,Supplies,0.077,-1189.10,190
4,Paper,0.075,34053.57,1370


**Hallazgo:** Tables y Bookcases tienen los descuentos promedio más altos (26% y 21%) y son las que más pierden. La hipótesis es que el descuento está destruyendo el margen.

## 4. Impacto del descuento en Tables por rango

In [10]:
query = '''
SELECT 
    CASE 
        WHEN discount = 0 THEN '0%'
        WHEN discount <= 0.15 THEN '1-15%'
        WHEN discount <= 0.30 THEN '16-30%'
        ELSE '31%+'
    END AS rango_descuento,
    COUNT(*) AS num_ordenes,
    ROUND(SUM(sales), 2) AS ventas,
    ROUND(SUM(profit), 2) AS ganancia
FROM orders
WHERE sub_category = 'Tables'
GROUP BY rango_descuento
ORDER BY MIN(discount);
'''

df_tables = pd.read_sql(query, conn)
df_tables

,rango_descuento,num_ordenes,ventas,ganancia
0,0%,72,71578.76,13276.30
1,16-30%,125,70612.38,-3705.89
2,31%+,122,64774.39,-27295.90


**Hallazgo:** Con ventas casi idénticas en los tres rangos, la ganancia cae drásticamente a medida que sube el descuento. Tables sin descuento genera +$13.276, con descuento >30% genera -$27.296.

## 5. Ventas y ganancia por región

In [11]:
query = '''
SELECT 
    region,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct,
    COUNT(DISTINCT order_id) AS num_ordenes
FROM orders
GROUP BY region
ORDER BY total_ventas DESC;
'''

df_region = pd.read_sql(query, conn)
df_region

,region,total_ventas,total_ganancia,margen_pct,num_ordenes
0,West,725457.82,108418.45,14.94,1611
1,East,678781.24,91522.78,13.48,1401
2,Central,501239.89,39706.36,7.92,1175
3,South,391721.91,46749.43,11.93,822


**Hallazgo:** West es la región más rentable (14.9% de margen). Central tiene el margen más bajo (7.92%) a pesar de vender más que South.

## 6. Descuento promedio por región

In [12]:
query = '''
SELECT 
    region,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia_total
FROM orders
GROUP BY region
ORDER BY descuento_promedio DESC;
'''

df_descuento_region = pd.read_sql(query, conn)
df_descuento_region

,region,descuento_promedio,ganancia_total
0,Central,0.240,39706.36
1,South,0.147,46749.43
2,East,0.145,91522.78
3,West,0.109,108418.45


**Hallazgo:** Patrón claro — a mayor descuento promedio, menor ganancia. Central tiene 24% de descuento promedio, casi el doble que West (10.9%).

## 7. Sub-categorías con peor desempeño en la región Central

In [13]:
query = '''
SELECT 
    sub_category,
    ROUND(AVG(discount), 3) AS descuento_promedio,
    ROUND(SUM(profit), 2) AS ganancia
FROM orders
WHERE region = 'Central'
GROUP BY sub_category
ORDER BY ganancia ASC
LIMIT 5;
'''

df_central = pd.read_sql(query, conn)
df_central

,sub_category,descuento_promedio,ganancia
0,Furnishings,0.404,-3906.22
1,Tables,0.262,-3559.65
2,Appliances,0.449,-2638.62
3,Bookcases,0.233,-1997.90
4,Machines,0.329,-1486.07


**Hallazgo:** En Central, incluso sub-categorías rentables a nivel nacional (Appliances, Furnishings) son negativas por descuentos de entre 33% y 45%. El problema de Central es generalizado.

## 8. Evolución anual de ventas y ganancia

In [14]:
query = '''
SELECT 
    strftime('%Y', order_date) AS anio,
    ROUND(SUM(sales), 2) AS total_ventas,
    ROUND(SUM(profit), 2) AS total_ganancia,
    ROUND(SUM(profit) * 100.0 / SUM(sales), 2) AS margen_pct
FROM orders
GROUP BY anio
ORDER BY anio;
'''

df_anual = pd.read_sql(query, conn)
df_anual

,anio,total_ventas,total_ganancia,margen_pct
0,2014,484247.50,49543.97,10.23
1,2015,470532.51,61618.60,13.10
2,2016,609205.60,81795.17,13.43
3,2017,733215.26,93439.27,12.74


**Hallazgo:** Las ventas casi se duplicaron entre 2014 y 2017 ($484K → $733K), pero el margen se estanca entre 10-13%. Esto indica que los problemas de descuento son crónicos y no se han corregido en 4 años.

In [16]:
# Cerrar la conexión al terminar
conn.close()

## Conclusiones

El análisis permitió identificar tres hallazgos principales:

**1. El crecimiento del negocio coexiste con ineficiencias estructurales.** Las ventas casi se duplicaron entre 2014 y 2017, pero el margen no mejoró, lo que indica que el crecimiento se está logrando a costa de descuentos que erosionan la rentabilidad.

**2. Los descuentos por encima del 30% son sistemáticamente destructivos de valor.** Esto se observa de forma clara en la sub-categoría Tables y en la región Central. En ambos casos, el volumen de ventas no compensa las pérdidas generadas por los descuentos excesivos.

**3. La región Central requiere atención prioritaria.** Su política de descuentos generalizada afecta incluso a categorías que son rentables en el resto del país.

**Recomendación:** establecer un límite máximo de descuento del 15-20% en las sub-categorías Tables y Bookcases, y revisar la estrategia comercial de la región Central. Según los datos, corregir el nivel de descuentos en Tables por sí solo podría transformar una pérdida de $17.725 en una ganancia de más de $13.000, con el mismo volumen de ventas.